<a href="https://colab.research.google.com/github/Enaganti2349/traffic-congestion-ml/blob/main/traffic_congestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

print("=== START PIPELINE ===")

# Load dataset
df = pd.read_csv('/content/traffic.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# Detect vehicle column
vehicle_col = None
for col in df.columns:
    if 'veh' in col.lower():
        vehicle_col = col
        break

if vehicle_col is None:
    raise ValueError("Vehicle column not found!")

print("Detected vehicle column:", vehicle_col)

# Convert DateTime
if 'DateTime' in df.columns:
    df['DateTime'] = pd.to_datetime(df['DateTime'], errors='coerce')
    df['hour'] = df['DateTime'].dt.hour
    df['day_of_week'] = df['DateTime'].dt.day_name()
else:
    df['hour'] = 0
    df['day_of_week'] = "Unknown"

# Create congestion categories using quantiles
q1 = df[vehicle_col].quantile(0.33)
q2 = df[vehicle_col].quantile(0.66)

def categorize(v):
    if v <= q1:
        return "low"
    elif v <= q2:
        return "medium"
    else:
        return "high"

df['congestion_level'] = df[vehicle_col].apply(categorize)

# Feature selection
features = ['hour']
if 'Junction' in df.columns:
    features.append('Junction')

features.append('day_of_week')

df_model = df[features + ['congestion_level']]

# One-hot encoding
df_model = pd.get_dummies(df_model, columns=['day_of_week'], drop_first=True)

X = df_model.drop('congestion_level', axis=1)
y = df_model['congestion_level']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, max_depth=12, random_state=42)
}

results = {}

# Train & Evaluate
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print("\n==============================")
    print("Model:", name)
    print("Accuracy:", round(acc, 4))
    print(classification_report(y_test, preds))

# Select Best Model
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("\n==============================")
print("Best Model:", best_model_name)
print("Best Accuracy:", round(results[best_model_name], 4))

# Save model
pickle.dump(best_model, open("traffic_model.pkl", "wb"))
pickle.dump(scaler, open("traffic_scaler.pkl", "wb"))

print("\nModel saved successfully!")
print("=== PIPELINE COMPLETE ===")

=== START PIPELINE ===
Shape: (48120, 4)
Columns: ['DateTime', 'Junction', 'Vehicles', 'ID']
Detected vehicle column: Vehicles

Model: Logistic Regression
Accuracy: 0.652
              precision    recall  f1-score   support

           0       0.74      0.82      0.78      3199
           1       0.66      0.73      0.70      3539
           2       0.49      0.37      0.42      2886

    accuracy                           0.65      9624
   macro avg       0.63      0.64      0.63      9624
weighted avg       0.64      0.65      0.64      9624


Model: KNN
Accuracy: 0.664
              precision    recall  f1-score   support

           0       0.72      0.79      0.76      3199
           1       0.71      0.75      0.73      3539
           2       0.50      0.42      0.46      2886

    accuracy                           0.66      9624
   macro avg       0.65      0.65      0.65      9624
weighted avg       0.65      0.66      0.66      9624


Model: Decision Tree
Accuracy: 0.7128


In [ ]:
!pip install pandas numpy scikit-learn matplotlib